# Assignment 10: Using LSTM for Future Weather Prediction

**Name:** Abdul Shamshuddin Sheikh  
**Class:** TY CSAI - B  
**Roll No:** 35  
**PRN:** 12310838

This notebook automatically downloads the dataset online, preprocesses it, trains an LSTM model, predicts future temperatures, and visualizes the results.


In [ ]:

# ==========================================
# Import Required Libraries
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input

print("Libraries Imported Successfully")


In [ ]:

# ==========================================
# Load Dataset Automatically
# ==========================================

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"

df = pd.read_csv(url)

print("Dataset Loaded Successfully")
print(df.head())

# Use only temperature column
data = df['Temp'].values.reshape(-1, 1)

print("\nDataset Shape:", data.shape)


In [ ]:

# ==========================================
# Data Preprocessing
# ==========================================

# Normalize data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Create sequence dataset
def create_dataset(data, time_step=30):
    X, y = [], []
    
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
    
    return np.array(X), np.array(y)

time_step = 30

X, y = create_dataset(scaled_data, time_step)

# Reshape for LSTM
X = X.reshape(X.shape[0], X.shape[1], 1)

# Train-test split
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Preprocessing Completed")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)


In [ ]:

# ==========================================
# Build LSTM Model
# ==========================================

model = Sequential([
    Input(shape=(time_step, 1)),
    LSTM(50, return_sequences=True),
    LSTM(50),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model.summary()


In [ ]:

# ==========================================
# Train Model
# ==========================================

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)


In [ ]:

# ==========================================
# Predictions
# ==========================================

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Inverse transform
train_pred = scaler.inverse_transform(train_pred)
test_pred = scaler.inverse_transform(test_pred)

y_train_actual = scaler.inverse_transform(y_train.reshape(-1,1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

print("Prediction Completed")


In [ ]:

# ==========================================
# Model Evaluation
# ==========================================

train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_actual, test_pred))

print("Train RMSE:", train_rmse)
print("Test RMSE :", test_rmse)


In [ ]:

# ==========================================
# Visualization: Actual vs Predicted
# ==========================================

plt.figure(figsize=(12,6))

plt.plot(y_test_actual, label='Actual Temperature')
plt.plot(test_pred, label='Predicted Temperature')

plt.title('Weather Prediction using LSTM')
plt.xlabel('Time')
plt.ylabel('Temperature')
plt.legend()

plt.show()


In [ ]:

# ==========================================
# Training Loss Graph
# ==========================================

plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('Loss Graph')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()


In [ ]:

# ==========================================
# Conclusion
# ==========================================

print("LSTM Weather Prediction Model Executed Successfully")
